In [1]:
pip install -U langchain langchain-core langchain-community langchain-openai langchain-groq langchain-cohere langchain-text-splitters chromadb python-dotenv beautifulsoup4 tiktoken

  Using cached langchain-1.3.8-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_openai-1.3.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached openai-2.41.1-py3-none-any.whl.metadata (32 kB)
Using cached langchain-1.3.8-py3-none-any.whl (132 kB)
Using cached langchain_openai-1.3.0-py3-none-any.whl (99 kB)
Using cached openai-2.41.1-py3-none-any.whl (1.4 MB)

  Attempting uninstall: beautifulsoup4

    Found existing installation: beautifulsoup4 4.14.3

   ---------------------------------------- 0/6 [beautifulsoup4]
    Uninstalling beautifulsoup4-4.14.3:
   ---------------------------------------- 0/6 [beautifulsoup4]
      Successfully uninstalled beautifulsoup4-4.14.3
   ---------------------------------------- 0/6 [beautifulsoup4]
   ---------------------------------------- 0/6 [beautifulsoup4]
   ---------------------------------------- 0/6 [beautifulsoup4]
   ---------------------------------------- 0/6 [beautifulsoup4]
   ---------------------------------------- 0/6 [

In [4]:
pip install -U openai langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [5]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

urls = [
    "https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-2-reflection/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-3-tool-use/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-4-planning/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-5-multi-agent-collaboration/?ref=dl-staging-website.ghost.io",
]

documents = []

for url in urls:
    loader = WebBaseLoader(url)
    documents.extend(loader.load())

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [8]:
question = "what are the differnt kind of agentic design patterns?"

In [9]:
docs = retriever.invoke(question)

In [10]:
print(f"Title: {docs[0].metadata['title']}\n\nSource: {docs[0].metadata['source']}\n\nContent: {docs[0].page_content}\n")

Title: Agentic Design Patterns Part 3: Tool Use

Source: https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-3-tool-use/?ref=dl-staging-website.ghost.io

Content: I’ll describe the Planning and Multi-agent collaboration design patterns. They allow AI agents to do much more but are less mature, less predictable — albeit very exciting — technologies. Keep learning!AndrewRead "Agentic Design Patterns Part 1: Four AI agent strategies that improve GPT-4 and GPT-3.5 performance"Read "Agentic Design Patterns Part 2: Reflection"Read "Agentic Design Patterns Part 4: Planning"Read "Agentic Design Patterns Part 5: Multi-Agent Collaboration"ShareSubscribe to The BatchStay updated with weekly AI News and Insights delivered to your inboxCoursesThe BatchCommunityCareersAboutContactHelpTerms of Use|Privacy Policy



In [12]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# Structured output schema
class GradeDocuments(BaseModel):
    """Determine whether a retrieved document is relevant."""

    binary_score: str = Field(
        description="Return only 'yes' or 'no'"
    )


from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

# Structured output wrapper
retrieval_grader_llm = llm.with_structured_output(
    GradeDocuments
)

# Prompt
grade_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a retrieval grader.

Your task is to determine whether a retrieved document is relevant to a user's question.

Rules:
- If the document contains information related to the question, return 'yes'.
- If the document is unrelated, return 'no'.
- Be lenient. The goal is to filter obviously irrelevant documents.
- Return only yes or no.
""",
        ),
        (
            "human",
            """
Retrieved document:

{document}

User question:

{question}
""",
        ),
    ]
)

# Chain
retrieval_grader = grade_prompt | retrieval_grader_llm

In [14]:
##Filter out the non-relevant docs

In [13]:
docs_to_use = []
for doc in docs:
    print(doc.page_content, '\n', '-'*50)
    res = retrieval_grader.invoke({"question": question, "document": doc.page_content})
    print(res,'\n')
    if res.binary_score == 'yes':
        docs_to_use.append(doc)

I’ll describe the Planning and Multi-agent collaboration design patterns. They allow AI agents to do much more but are less mature, less predictable — albeit very exciting — technologies. Keep learning!AndrewRead "Agentic Design Patterns Part 1: Four AI agent strategies that improve GPT-4 and GPT-3.5 performance"Read "Agentic Design Patterns Part 2: Reflection"Read "Agentic Design Patterns Part 4: Planning"Read "Agentic Design Patterns Part 5: Multi-Agent Collaboration"ShareSubscribe to The BatchStay updated with weekly AI News and Insights delivered to your inboxCoursesThe BatchCommunityCareersAboutContactHelpTerms of Use|Privacy Policy 
 --------------------------------------------------
binary_score='yes' 

up tasks and discussing and debating ideas, to come up with better solutions than a single agent would.Next week, I’ll elaborate on these design patterns and offer suggested readings for each.Keep learning!AndrewRead "Agentic Design Patterns Part 2: Reflection"Read "Agentic Desig

In [15]:
from langchain_core.output_parsers import StrOutputParser

# Prompt
system = """You are an assistant for question-answering tasks. Answer the question based upon your knowledge. 
Use three-to-five sentences maximum and keep the answer concise."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved documents: \n\n <docs>{documents}</docs> \n\n User question: <question>{question}</question>"),
    ]
)

# LLM
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

# Post-processing
def format_docs(docs):
    return "\n".join(f"<doc{i+1}>:\nTitle:{doc.metadata['title']}\nSource:{doc.metadata['source']}\nContent:{doc.page_content}\n</doc{i+1}>\n" for i, doc in enumerate(docs))

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"documents":format_docs(docs_to_use), "question": question})
print(generation)

The different kinds of agentic design patterns are Reflection, Tool Use, Planning, and Multi-agent Collaboration. Reflection involves the LLM examining its own work to improve it. Tool Use allows the LLM to use external tools like web search or code execution. Planning enables the LLM to create and execute multistep plans to achieve goals. Multi-agent Collaboration involves multiple AI agents working together, dividing tasks and discussing ideas for better solutions.


In [16]:
##Check for Hallucinations

In [17]:
# Data model
class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in 'generation' answer."""

    binary_score: str = Field(
        ...,
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)
structured_llm_grader = llm.with_structured_output(GradeHallucinations)

# Prompt
system = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
    Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""
hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Set of facts: \n\n <facts>{documents}</facts> \n\n LLM generation: <generation>{generation}</generation>"),
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_grader

response = hallucination_grader.invoke({"documents": format_docs(docs_to_use), "generation": generation})
print(response)

binary_score='yes'


In [18]:
##Highlight used docs

In [20]:
from typing import List
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

# Data model
class HighlightDocuments(BaseModel):
    """Return the specific part of a document used for answering the question."""

    id: List[str] = Field(
        ...,
        description="List of id of docs used to answers the question"
    )

    title: List[str] = Field(
        ...,
        description="List of titles used to answers the question"
    )

    source: List[str] = Field(
        ...,
        description="List of sources used to answers the question"
    )

    segment: List[str] = Field(
        ...,
        description="List of direct segements from used documents that answers the question"
    )

# LLM
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

# parser
parser = PydanticOutputParser(pydantic_object=HighlightDocuments)

# Prompt
system = """You are an advanced assistant for document search and retrieval. You are provided with the following:
1. A question.
2. A generated answer based on the question.
3. A set of documents that were referenced in generating the answer.

Your task is to identify and extract the exact inline segments from the provided documents that directly correspond to the content used to 
generate the given answer. The extracted segments must be verbatim snippets from the documents, ensuring a word-for-word match with the text 
in the provided documents.

Ensure that:
- (Important) Each segment is an exact match to a part of the document and is fully contained within the document text.
- The relevance of each segment to the generated answer is clear and directly supports the answer provided.
- (Important) If you didn't used the specific document don't mention it.

Used documents: <docs>{documents}</docs> \n\n User question: <question>{question}</question> \n\n Generated answer: <answer>{generation}</answer>

<format_instruction>
{format_instructions}
</format_instruction>
"""


prompt = PromptTemplate(
    template= system,
    input_variables=["documents", "question", "generation"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Chain
doc_lookup = prompt | llm | parser

# Run
lookup_response = doc_lookup.invoke({"documents":format_docs(docs_to_use), "question": question, "generation": generation})

In [21]:
for id, title, source, segment in zip(lookup_response.id, lookup_response.title, lookup_response.source, lookup_response.segment):
    print(f"ID: {id}\nTitle: {title}\nSource: {source}\nText Segment: {segment}\n")

ID: doc3
Title: Four AI Agent Strategies That Improve GPT-4 and GPT-3.5 Performance
Source: https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/?ref=dl-staging-website.ghost.io
Text Segment: Reflection: The LLM examines its own work to come up with ways to improve it. Tool Use: The LLM is given tools such as web search, code execution, or any other function to help it gather information, take action, or process data. Planning: The LLM comes up with, and executes, a multistep plan to achieve a goal (for example, writing an outline for an essay, then doing online research, then writing a draft, and so on). Multi-agent collaboration: More than one AI agent work together, splitting up tasks and discussing and debating ideas, to come up with better solutions than a single agent would.

